# Silver Layer — Cleaning, Casting & Deduplication

In [0]:
from pyspark.sql import functions as F

In [0]:
df_prices_bronze = spark.table("bronze_prices")

df_silver_prices = df_prices_bronze.select(
    F.col("id"),
    F.col("symbol"),
    F.col("name"),
    F.col("current_price").cast("double"),
    F.col("market_cap").cast("double"),
    F.col("market_cap_rank").cast("integer"),
    F.col("total_volume").cast("double"),
    F.col("high_24h").cast("double"),
    F.col("low_24h").cast("double"),
    F.col("price_change_24h").cast("double"),
    F.col("price_change_percentage_24h").cast("double"),
    F.col("circulating_supply").cast("double"),
    F.col("last_updated").cast("timestamp")
).dropna(
    subset=["id", "current_price", "last_updated"]
).dropDuplicates(
    ["id", "last_updated"]
)

print(f"Silver prices rows: {df_silver_prices.count()}")
display(df_silver_prices)

In [0]:
df_fg_bronze = spark.table("bronze_fear_greed")

df_silver_fg = df_fg_bronze.select(
    F.col("ingested_at").cast("timestamp").alias("ingested_at"),
    F.col("source"),
    F.get_json_object(F.col("data"), "$.data[0].value").cast("integer").alias("fear_greed_value"),
    F.get_json_object(F.col("data"), "$.data[0].value_classification").alias("value_classification"),
    F.get_json_object(F.col("data"), "$.data[0].timestamp").cast("long").alias("fg_unix_timestamp")
).dropna(
    subset=["ingested_at", "fear_greed_value"]
).dropDuplicates(
    ["ingested_at"]
)

print(f"Silver fear & greed rows: {df_silver_fg.count()}")
display(df_silver_fg)